In [1]:
!pip3 install selenium webdriver-manager pandas openpyxl

In [2]:
!pip3 install scheduler

In [3]:
import pandas as pd
import openpyxl
from typing import Optional

def _get_link_if_exists(cell) -> Optional[str]:
    try:
        return cell.hyperlink.target
    except AttributeError:
        return None


def extract_hyperlinks_from_xlsx(file_path, sheet_name=None):
    wb = load_workbook(filename=file_path, data_only=True)

    # 👉 Nếu không truyền sheet_name hoặc sheet không tồn tại → lấy sheet đầu tiên
    if not sheet_name or sheet_name not in wb.sheetnames:
        ws = wb.worksheets[0]
        print(f"⚠️ Sheet '{sheet_name}' không tồn tại → dùng sheet mặc định: '{ws.title}'")
    else:
        ws = wb[sheet_name]

    urls = []
    hotel_names = []
    room_types = []

    for row in ws.iter_rows(min_row=2, max_col=2):
        hotel_cell = row[0]
        room_cell = row[1]

        hotel_names.append(hotel_cell.value)
        urls.append(hotel_cell.hyperlink.target if hotel_cell.hyperlink else "")
        room_types.append(room_cell.value)

    return pd.DataFrame({
        "Hotel": urls,
        "Hotel_name": hotel_names,
        "Room_type": room_types
    })


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import random
from datetime import datetime, timedelta, date
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import os

# ============================================================
# CONFIG - STEALTH MODE (Tránh bị chặn)
# ============================================================
# 📁 FILE SETTINGS
INPUT_FILE = "./raw.csv"               # File CSV chứa danh sách hotels (3 columns: hotel_name, hotel_url, room_type)
TEMP_OUTPUT_FILE = "hotel_prices_temp.csv"  # File lưu tạm khi đang crawl
OUTPUT_PREFIX = "hotel_prices_"        # Prefix cho file output cuối cùng (sẽ thêm _YYYYMMDD.csv)

# ⚙️ PERFORMANCE SETTINGS - STEALTH MODE
NUM_WORKERS = 4          # ↓ Giảm từ 6 → 4 (ít requests hơn = ít bị chặn hơn)
WEEKS_PER_HOTEL = 2      # ↓ Giảm từ 3 → 2 (giảm Chrome instances đồng thời)
MAX_RETRIES = 3          # ↑ Tăng từ 2 → 3 (retry nhiều hơn nếu bị chặn)
DELAY_RANGE = (1.5, 3.5) # ↑ Tăng từ (0.3, 0.8) → (1.5, 3.5) - delay dài hơn
HOTEL_DELAY = (3, 6)     # ← MỚI: Delay giữa mỗi hotel (giây)
PAGE_TIMEOUT = 20        # ↑ Tăng từ 15 → 20 (chờ lâu hơn)

# ============================================================
# HÀM ĐỌC CSV - ĐƠN GIẢN HƠN NHIỀU
# ============================================================
def read_hotels_from_csv(file_path):
    """
    Đọc danh sách hotels từ file CSV
    Expected columns: hotel_name, hotel_url, room_type
    """
    try:
        df = pd.read_csv(file_path)
        
        # Check columns
        required_cols = ['hotel_name', 'hotel_url', 'room_type']
        
        # Nếu không có tên chuẩn, thử dùng columns theo thứ tự
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️  CSV không có columns chuẩn. Dùng 3 columns đầu tiên.")
            print(f"   Columns hiện tại: {list(df.columns)}")
            
            # Rename 3 columns đầu tiên
            df.columns = ['hotel_name', 'hotel_url', 'room_type'] + list(df.columns[3:])
        
        # Lọc bỏ rows không có URL
        df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
        
        print(f"✅ Đọc được {len(df)} hotels từ {file_path}")
        return df[['hotel_name', 'hotel_url', 'room_type']]
        
    except Exception as e:
        print(f"❌ Lỗi đọc file CSV: {e}")
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

# ============================================================
# TẠO CHROME DRIVER - STEALTH MODE
# ============================================================
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0",
]

def create_driver():
    """Tạo Chrome driver với stealth settings"""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    # Stealth: Enable images để giống browser thật hơn
    options.add_experimental_option("prefs", {
        "profile.default_content_setting_values.notifications": 2,
        "credentials_enable_service": False,
        "profile.password_manager_enabled": False
    })
    
    options.page_load_strategy = "normal"  # Đổi từ "eager" → "normal" để giống user thật
    options.add_argument(f"user-agent={random.choice(USER_AGENTS)}")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-logging")
    options.add_argument("--log-level=3")
    options.add_argument("--lang=en-US,en")
    
    driver = webdriver.Chrome(options=options)
    
    # Stealth: Ẩn webdriver property
    driver.execute_cdp_cmd('Network.setUserAgentOverride', {
        "userAgent": random.choice(USER_AGENTS)
    })
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

# ============================================================
# HÀM SET NGÀY - STEALTH MODE
# ============================================================
def set_dates(driver, wait, checkin_date, checkout_date):
    def open_calendar():
        checkin_box = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-selenium='checkInBox']")))
        time.sleep(random.uniform(0.3, 0.7))
        checkin_box.click()
        time.sleep(random.uniform(0.5, 1.0))

    def click_next_month_until_visible(target_date):
        for _ in range(12):
            try:
                selector = f"[data-selenium-date='{target_date.strftime('%Y-%m-%d')}']"
                driver.find_element(By.CSS_SELECTOR, selector)
                return
            except:
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "[data-selenium='calendar-next-month-button']")
                    time.sleep(random.uniform(0.2, 0.5))
                    next_btn.click()
                    time.sleep(random.uniform(0.4, 0.8))
                except:
                    break

    def select_date(d):
        click_next_month_until_visible(d)
        selector = f"[data-selenium-date='{d.strftime('%Y-%m-%d')}']"
        date_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))
        driver.execute_script("arguments[0].scrollIntoView(true);", date_button)
        time.sleep(random.uniform(0.2, 0.5))
        date_button.click()
        time.sleep(random.uniform(0.3, 0.7))

    open_calendar()
    select_date(checkin_date)
    select_date(checkout_date)

# ============================================================
# HÀM CÀO GIÁ 1 TUẦN - STEALTH MODE
# ============================================================
def scrape_room_prices(driver, url, checkin_date, checkout_date):
    driver.get(url)
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    results = []

    try:
        time.sleep(random.uniform(1.0, 2.0))
        
        # Scroll ngẫu nhiên để giống user thật
        driver.execute_script(f"window.scrollTo(0, {random.randint(100, 300)});")
        time.sleep(random.uniform(0.3, 0.7))
        driver.execute_script("window.scrollTo(0, 0);")
        
        try:
            close_btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, ".ab-close-button"))
            )
            time.sleep(random.uniform(0.2, 0.5))
            close_btn.click()
            time.sleep(random.uniform(0.3, 0.6))
        except:
            pass

        set_dates(driver, wait, checkin_date, checkout_date)

        search_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-selenium='searchButton']")))
        time.sleep(random.uniform(0.5, 1.0))
        search_btn.click()

        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div#roomGrid")))
        time.sleep(random.uniform(1.0, 2.0))
        
        # Scroll giống user thật
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        time.sleep(random.uniform(0.5, 1.0))
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(0.8, 1.5))

        room_cards = driver.find_elements(By.CSS_SELECTOR, "div[data-selenium='MasterRoom']")
        for card in room_cards:
            try:
                name = card.find_element(By.CSS_SELECTOR, "[data-selenium='masterroom-title-name']").text
            except:
                name = "NA"
            try:
                price = card.find_element(By.CSS_SELECTOR, "[data-selenium='PriceDisplay']").text
            except:
                price = "NA"
            results.append({"room": name, "price": price})
    except Exception as e:
        pass

    return results

# ============================================================
# LƯU DATA RA CSV (THREAD-SAFE) - WITH DEBUG
# ============================================================
def save_backup_csv(all_week_prices, filename, lock):
    """Lưu toàn bộ data hiện có ra file CSV"""
    try:
        with lock:
            rows = []
            for (hotel, room), prices in all_week_prices.items():
                row = {"hotel_name": hotel, "room_type": room}
                for i in range(1, 7):
                    row[f"price_w{i}"] = prices.get(f"Price W{i}", "NA")
                rows.append(row)
            
            df = pd.DataFrame(rows)
            df.to_csv(filename, index=False)
            print(f"💾 Saved {len(rows)} hotels to {filename}")
    except Exception as e:
        print(f"❌ Error saving to {filename}: {e}")

# ============================================================
# CRAWL 1 TUẦN - SUB-WORKER (STEALTH)
# ============================================================
def crawl_single_week(hotel_url, room_type, week_num, checkin, checkout):
    """Crawl giá cho 1 tuần cụ thể - chạy song song"""
    driver = create_driver()
    result = {"week": week_num, "price": "NA", "date": checkin.strftime('%Y-%m-%d')}
    
    try:
        for retry in range(MAX_RETRIES):
            check_in = checkin + timedelta(days=retry)
            check_out = checkout + timedelta(days=retry)
            
            if retry > 0:
                backoff_delay = random.uniform(2, 4) * (retry + 1)
                print(f"      🔄 W{week_num} retry {retry}/{MAX_RETRIES} (waiting {backoff_delay:.1f}s)")
                time.sleep(backoff_delay)
            
            try:
                data = scrape_room_prices(driver, hotel_url, check_in, check_out)
                
                for room_data in data:
                    if room_data["room"].strip() == room_type.strip() and room_data["price"] != "NA":
                        result["price"] = room_data["price"]
                        result["date"] = check_in.strftime('%Y-%m-%d')
                        print(f"      ✅ W{week_num}: {room_data['price']} | {check_in.strftime('%Y-%m-%d')}")
                        driver.quit()
                        return result
                
                if retry < MAX_RETRIES - 1:
                    time.sleep(random.uniform(*DELAY_RANGE))
                    
            except Exception as e:
                if retry < MAX_RETRIES - 1:
                    print(f"      ⚠️  W{week_num} error, retrying...")
                continue
        
        print(f"      ❌ W{week_num}: NA")
    finally:
        try:
            driver.quit()
        except:
            pass
    
    return result

# ============================================================
# WORKER: XỬ LÝ 1 HOTEL - STEALTH MODE
# ============================================================
def process_hotel(hotel_info, prev_data, base_checkin, base_checkout, week_offsets):
    hotel_name, hotel_url, room_type = hotel_info
    key = (hotel_name, room_type)

    # Skip nếu đã có đủ data
    if key in prev_data:
        all_cached = all(
            prev_data[key].get(f"Price W{i}", "NA") != "NA"
            for i in range(1, 7)
        )
        if all_cached:
            print(f"⏭️  SKIP: {hotel_name} | {room_type}")
            return key, prev_data[key], True

    # Delay trước khi bắt đầu hotel mới
    time.sleep(random.uniform(*HOTEL_DELAY))
    
    print(f"\n🏨 START: {hotel_name} | Room: {room_type}")
    prices = {}

    # Tìm các tuần cần crawl
    weeks_to_crawl = []
    for week_num, offset in enumerate(week_offsets, start=1):
        key_prefix = f"Price W{week_num}"
        
        # Sử dụng cached data nếu có
        if key in prev_data and prev_data[key].get(key_prefix, "NA") != "NA":
            cached_price = prev_data[key][key_prefix]
            prices[key_prefix] = cached_price
            print(f"   ✓ W{week_num}: {cached_price} (cached)")
        else:
            weeks_to_crawl.append((week_num, offset))

    # CRAWL SONG SONG CÁC TUẦN CÒN THIẾU
    if weeks_to_crawl:
        print(f"   🚀 Crawling {len(weeks_to_crawl)} weeks (max {WEEKS_PER_HOTEL} parallel)...")
        
        with ThreadPoolExecutor(max_workers=min(len(weeks_to_crawl), WEEKS_PER_HOTEL)) as week_executor:
            week_futures = {}
            
            for week_num, offset in weeks_to_crawl:
                checkin = base_checkin + timedelta(days=offset)
                checkout = base_checkout + timedelta(days=offset)
                
                future = week_executor.submit(
                    crawl_single_week, 
                    hotel_url, 
                    room_type, 
                    week_num, 
                    checkin, 
                    checkout
                )
                week_futures[future] = week_num
            
            # Thu thập kết quả
            for future in as_completed(week_futures):
                result = future.result()
                prices[f"Price W{result['week']}"] = result["price"]

    print(f"✅ DONE: {hotel_name}")
    return key, prices, False

# ============================================================
# MAIN
# ============================================================
thread_local = threading.local()
save_lock = threading.Lock()

if __name__ == "__main__":
    start_time = time.time()
    
    # ĐỌC CSV
    df_hotels = read_hotels_from_csv(INPUT_FILE)
    
    if len(df_hotels) == 0:
        print("❌ Không có hotels nào để crawl!")
        exit()

    # Load data cũ từ CSV
    all_week_prices = {}
    prev_data = {}

    if os.path.exists(TEMP_OUTPUT_FILE):
        print(f"📂 Đọc file tạm: {TEMP_OUTPUT_FILE}")
        df_prev = pd.read_csv(TEMP_OUTPUT_FILE)
        for _, row in df_prev.iterrows():
            key = (row["hotel_name"], row["room_type"])
            prices = {f"Price W{i}": row.get(f"price_w{i}", "NA") for i in range(1, 7)}
            prev_data[key] = prices
            all_week_prices[key] = prices
        print(f"✅ Đã load {len(prev_data)} hotels từ lần chạy trước")

    base_checkin = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
    base_checkout = base_checkin + timedelta(days=1)
    week_offsets = [0, 7, 14, 21, 28, 35]

    # Chuẩn bị tasks
    hotel_tasks = []
    skip_count = 0
    
    for _, row in df_hotels.iterrows():
        hotel_name = row['hotel_name']
        hotel_url = row['hotel_url']
        room_type = row['room_type']
        
        key = (hotel_name, room_type)
        
        # Đếm hotels đã hoàn thành
        if key in prev_data and all(prev_data[key].get(f"Price W{i}", "NA") != "NA" for i in range(1, 7)):
            skip_count += 1
        
        hotel_tasks.append((hotel_name, hotel_url, room_type))

    need_crawl = len(hotel_tasks) - skip_count
    print(f"\n{'='*60}")
    print(f"🥷 STEALTH MODE - Anti-Detection Crawling")
    print(f"📊 Tổng: {len(hotel_tasks)} hotels")
    print(f"⏭️  Skip: {skip_count} (đã có data)")
    print(f"🔄 Cần crawl: {need_crawl}")
    print(f"⚡ Workers: {NUM_WORKERS} hotels × {WEEKS_PER_HOTEL} weeks (slower but safer)")
    print(f"⏱️  Delays: {DELAY_RANGE[0]}-{DELAY_RANGE[1]}s/request, {HOTEL_DELAY[0]}-{HOTEL_DELAY[1]}s/hotel")
    print(f"{'='*60}\n")

    completed = 0
    crawled = 0

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {
            executor.submit(process_hotel, task, prev_data, base_checkin, base_checkout, week_offsets): task
            for task in hotel_tasks
        }

        for future in as_completed(futures):
            task = futures[future]
            try:
                key, prices, skipped = future.result()

                with save_lock:
                    all_week_prices[key] = prices

                completed += 1

                if not skipped:
                    crawled += 1

                    status = " | ".join([
                        f"W{i}:{'✓' if prices.get(f'Price W{i}', 'NA') != 'NA' else '✗'}"
                        for i in range(1, 7)
                    ])
                    elapsed = time.time() - start_time
                    rate = elapsed / max(crawled, 1)
                    remaining = need_crawl - crawled
                    eta = rate * remaining
                    print(f"\n📊 [{crawled}/{need_crawl}] {key[0]} | {status} | ETA: {int(eta//60)}m{int(eta%60)}s")

                    # Save ngay sau mỗi hotel
                    save_backup_csv(all_week_prices, TEMP_OUTPUT_FILE, save_lock)

            except Exception as e:
                print(f"❌ Lỗi {task[0]}: {e}")
                completed += 1

    # Xuất file chính thức
    final_filename = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
    save_backup_csv(all_week_prices, final_filename, save_lock)

    total_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✅ HOÀN THÀNH!")
    print(f"📁 Saved to: {final_filename}")
    print(f"📊 Tổng hotels: {len(all_week_prices)}")
    print(f"⏱️  Thời gian: {int(total_time//60)} phút {int(total_time%60)} giây")
    if crawled > 0:
        print(f"⚡ Tốc độ: {total_time/crawled:.1f}s / hotel")
        print(f"🚀 Tổng requests: {crawled * 6} weeks crawled")
    print(f"{'='*60}")

/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


⚠️  CSV không có columns chuẩn. Dùng 3 columns đầu tiên.
   Columns hiện tại: ['Hotel', 'URL', 'Room to be extracted']
✅ Đọc được 229 hotels từ ./raw.csv
📂 Đọc file tạm: hotel_prices_temp.csv
✅ Đã load 3 hotels từ lần chạy trước

🥷 STEALTH MODE - Anti-Detection Crawling
📊 Tổng: 229 hotels
⏭️  Skip: 3 (đã có data)
🔄 Cần crawl: 226
⚡ Workers: 4 hotels × 2 weeks (slower but safer)
⏱️  Delays: 1.5-3.5s/request, 3-6s/hotel

⏭️  SKIP: Becamex Hotel New City | Twin One Bedroom Suite
⏭️  SKIP: Citadines Central Binh Duong | Studio Premier
⏭️  SKIP: HIIVE by fusion Binh Duong - VSIP 1 | Superior Twin

🏨 START: Sheraton Can Tho | Room: Guest room, 2 Twin
   🚀 Crawling 6 weeks (max 2 parallel)...

🏨 START: Legacy Mekong | Room: Bungalow, 2 Twin, Garden View
   🚀 Crawling 6 weeks (max 2 parallel)...

🏨 START: Ana Mandara Villas Dalat Resort & Spa | Room: Villa Double Room
   🚀 Crawling 6 weeks (max 2 parallel)...

🏨 START: Fairfield by Marriott South Binh Duong | Room: Guest room, 2 Twin
   🚀 Craw